# Analisis Keterlambatan Penerbangan AS 2015 dengan PySpark
Midterm Project Big Data & AI. Notebook ini dijalankan berurutan dari atas ke bawah.

**Tahap 1: Setup Spark dan ingestion.** Semua angka (jumlah baris, dll.) dihitung oleh notebook ini, bukan disalin dari draf proposal.

## 1. Konfigurasi Spark
Satu sel konfigurasi: mode lokal memakai semua core, memori driver 8 GB, dan 16 partisi shuffle.

In [ ]:
# Membuat SparkSession dengan konfigurasi tetap (dipakai seluruh notebook)
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("flight_delays_2015")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())

# Mengurangi log agar output notebook tidak penuh
spark.sparkContext.setLogLevel("WARN")

## 2. Versi lingkungan
Versi dicatat agar penguji tahu lingkungan yang dipakai.

In [ ]:
# Mencetak versi Python, PySpark, dan Java, serta konfigurasi Spark penting
import sys
import pyspark

print("Python  :", sys.version)
print("PySpark :", pyspark.__version__)
print("Java    :", spark._jvm.System.getProperty("java.version"))
print("master  :", spark.sparkContext.master)
print("driver.memory      :", spark.conf.get("spark.driver.memory"))
print("shuffle.partitions :", spark.conf.get("spark.sql.shuffle.partitions"))

## 3. Membaca data dengan schema eksplisit
Schema ditulis sendiri (tanpa inferSchema). Kolom jam format HHMM dibaca sebagai **string** agar nol di depan (misalnya `0005`) tidak hilang.

In [ ]:
# Mengimpor tipe data untuk membuat schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [ ]:
# Schema flights.csv (31 kolom, urutan sama dengan header file)
schema_flights = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("AIRLINE", StringType(), True),
    StructField("FLIGHT_NUMBER", IntegerType(), True),
    StructField("TAIL_NUMBER", StringType(), True),
    StructField("ORIGIN_AIRPORT", StringType(), True),
    StructField("DESTINATION_AIRPORT", StringType(), True),
    StructField("SCHEDULED_DEPARTURE", StringType(), True),   # HHMM, string
    StructField("DEPARTURE_TIME", StringType(), True),        # HHMM, string
    StructField("DEPARTURE_DELAY", IntegerType(), True),
    StructField("TAXI_OUT", IntegerType(), True),
    StructField("WHEELS_OFF", StringType(), True),            # HHMM, string
    StructField("SCHEDULED_TIME", IntegerType(), True),
    StructField("ELAPSED_TIME", IntegerType(), True),
    StructField("AIR_TIME", IntegerType(), True),
    StructField("DISTANCE", IntegerType(), True),
    StructField("WHEELS_ON", StringType(), True),             # HHMM, string
    StructField("TAXI_IN", IntegerType(), True),
    StructField("SCHEDULED_ARRIVAL", StringType(), True),     # HHMM, string
    StructField("ARRIVAL_TIME", StringType(), True),          # HHMM, string
    StructField("ARRIVAL_DELAY", IntegerType(), True),
    StructField("DIVERTED", IntegerType(), True),
    StructField("CANCELLED", IntegerType(), True),
    StructField("CANCELLATION_REASON", StringType(), True),
    StructField("AIR_SYSTEM_DELAY", IntegerType(), True),
    StructField("SECURITY_DELAY", IntegerType(), True),
    StructField("AIRLINE_DELAY", IntegerType(), True),
    StructField("LATE_AIRCRAFT_DELAY", IntegerType(), True),
    StructField("WEATHER_DELAY", IntegerType(), True),
])

In [ ]:
# Schema airports.csv
schema_airports = StructType([
    StructField("IATA_CODE", StringType(), True),
    StructField("AIRPORT", StringType(), True),
    StructField("CITY", StringType(), True),
    StructField("STATE", StringType(), True),
    StructField("COUNTRY", StringType(), True),
    StructField("LATITUDE", DoubleType(), True),
    StructField("LONGITUDE", DoubleType(), True),
])

In [ ]:
# Schema airlines.csv
schema_airlines = StructType([
    StructField("IATA_CODE", StringType(), True),
    StructField("AIRLINE", StringType(), True),
])

In [ ]:
# Membaca flights.csv memakai schema eksplisit
df_flights = spark.read.csv("../data/flights.csv", header=True, schema=schema_flights)

In [ ]:
# Membaca airports.csv
df_airports = spark.read.csv("../data/airports.csv", header=True, schema=schema_airports)

In [ ]:
# Membaca airlines.csv
df_airlines = spark.read.csv("../data/airlines.csv", header=True, schema=schema_airlines)

## 4. Schema
Memastikan tipe kolom sesuai rencana (kolom jam bertipe string).

In [ ]:
# Schema flights
df_flights.printSchema()

In [ ]:
# Schema airports
df_airports.printSchema()

In [ ]:
# Schema airlines
df_airlines.printSchema()

## 5. Jumlah baris dan jumlah partisi
Jumlah baris dihitung dari data, lalu dibandingkan dengan dokumentasi sumber saat laporan disusun. Jumlah partisi menunjukkan bagaimana Spark membagi file saat dibaca.

In [ ]:
# Jumlah baris dan kolom tiap tabel
print("flights  :", df_flights.count(), "baris,", len(df_flights.columns), "kolom")
print("airports :", df_airports.count(), "baris,", len(df_airports.columns), "kolom")
print("airlines :", df_airlines.count(), "baris,", len(df_airlines.columns), "kolom")

In [ ]:
# Jumlah partisi saat data dibaca
print("partisi flights  :", df_flights.rdd.getNumPartitions())
print("partisi airports :", df_airports.rdd.getNumPartitions())
print("partisi airlines :", df_airlines.rdd.getNumPartitions())

## 6. Sampel data
Data berupa catatan per penerbangan (bukan per penumpang), jadi tidak ada data pribadi. Sampel kecil ditampilkan untuk memeriksa bahwa kolom jam masih berformat HHMM dengan nol di depan.

In [ ]:
# Lima baris pertama flights (kolom penting saja agar mudah dibaca)
df_flights.select("YEAR", "MONTH", "DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT",
                  "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE", "DEPARTURE_TIME",
                  "ARRIVAL_DELAY", "CANCELLED", "DIVERTED").show(5)

In [ ]:
# Lima baris pertama airports
df_airports.show(5, truncate=False)

In [ ]:
# Seluruh airlines (hanya sedikit baris)
df_airlines.show(truncate=False)

## 7. Menyimpan flights ke Parquet
CSV mentah tetap di `../data/` sebagai lapisan mentah. Salinan Parquet (kolumnar, schema tersimpan) disimpan di `../data/processed/flights` untuk dipakai tahap berikutnya. Mode `overwrite` supaya sel aman dijalankan ulang.

In [ ]:
# Menulis flights ke Parquet (folder data/processed dibuat otomatis oleh Spark)
df_flights.write.mode("overwrite").parquet("../data/processed/flights")

In [ ]:
# Verifikasi: baca ulang Parquet, jumlah baris harus sama dengan CSV
df_cek = spark.read.parquet("../data/processed/flights")
print("baris Parquet:", df_cek.count())
print("partisi Parquet:", df_cek.rdd.getNumPartitions())

# Tahap 2: Processing dengan PySpark
Definisi yang dipakai di seluruh tahap ini:
- **Delay** = `ARRIVAL_DELAY >= 15` menit (konvensi BTS).
- Penerbangan **valid** = tidak dibatalkan (`CANCELLED = 0`), tidak dialihkan (`DIVERTED = 0`), dan `ARRIVAL_DELAY` tidak null.
- Analisis **bandara/rute** hanya memakai baris yang kode bandaranya terpetakan ke `airports.csv`. Analisis lain (maskapai, bulan, hari, jam) memakai semua baris valid.

## A. Membaca ulang data dari Parquet
Mulai tahap ini data dibaca dari lapisan Parquet hasil Tahap 1.

In [ ]:
# Mengimpor fungsi Spark, window, dan modul waktu yang dipakai di Tahap 2
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

In [ ]:
# Membaca flights dari Parquet
df_flights_p = spark.read.parquet("../data/processed/flights")
print("jumlah baris:", df_flights_p.count())

## B. Menghitung ulang profil awal
Semua angka dihitung dari data, bukan disalin dari draf proposal. Satu kali pembacaan data (satu `agg`) dipakai untuk semua angka ringkasan.

In [ ]:
# Ringkasan profil: total, batal, dialihkan, ARRIVAL_DELAY terisi, delay >= 15,
# dan baris yang ARRIVAL_DELAY-nya terisi padahal batal/dialihkan (untuk memeriksa tumpang tindih)
df_profil = df_flights_p.agg(
    F.count(F.lit(1)).alias("total_baris"),
    F.sum("CANCELLED").alias("jumlah_batal"),
    F.sum("DIVERTED").alias("jumlah_dialihkan"),
    F.count("ARRIVAL_DELAY").alias("arrival_delay_terisi"),
    F.sum(F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0)).alias("jumlah_delay_15"),
    F.sum(F.when(F.col("ARRIVAL_DELAY").isNotNull() &
                 ((F.col("CANCELLED") == 1) | (F.col("DIVERTED") == 1)), 1).otherwise(0)).alias("terisi_tapi_batal_atau_alih")
)
df_profil.show(truncate=False)

In [ ]:
# Proporsi delay terhadap baris yang ARRIVAL_DELAY-nya terisi
baris_profil = df_profil.collect()[0]
print("proporsi delay >= 15 menit:", baris_profil["jumlah_delay_15"] / baris_profil["arrival_delay_terisi"])

In [ ]:
# Jumlah baris dengan kode bandara numerik per bulan (kode IATA berisi huruf, kode numerik hanya angka)
df_numerik = (df_flights_p
              .withColumn("asal_numerik", F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$").cast("int"))
              .withColumn("tujuan_numerik", F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$").cast("int"))
              .groupBy("MONTH")
              .agg(F.count(F.lit(1)).alias("total_baris"),
                   F.sum("asal_numerik").alias("asal_numerik"),
                   F.sum("tujuan_numerik").alias("tujuan_numerik"))
              .orderBy("MONTH"))
df_numerik.show(12)

## C. Operasi PySpark DataFrame API

### Operasi 1: Filter penerbangan valid dan kolom turunan
Dibuat `is_delayed` (1 jika `ARRIVAL_DELAY >= 15`) dan `jam_jadwal` (dua digit pertama `SCHEDULED_DEPARTURE`).

In [ ]:
# Filter penerbangan valid + kolom turunan is_delayed dan jam_jadwal
df_valid = (df_flights_p
            .filter((F.col("CANCELLED") == 0) &
                    (F.col("DIVERTED") == 0) &
                    F.col("ARRIVAL_DELAY").isNotNull())
            .withColumn("is_delayed", F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0))
            .withColumn("jam_jadwal", F.substring("SCHEDULED_DEPARTURE", 1, 2).cast("int")))

n_semua = df_flights_p.count()
n_valid = df_valid.count()
print("baris sebelum filter:", n_semua)
print("baris valid         :", n_valid)
print("baris dibuang       :", n_semua - n_valid)

In [ ]:
# Memeriksa panjang string SCHEDULED_DEPARTURE (seharusnya selalu 4 karakter, HHMM)
df_flights_p.groupBy(F.length("SCHEDULED_DEPARTURE").alias("panjang")).count().show()

In [ ]:
# Memeriksa apakah ada nilai "2400" dan rentang jam_jadwal, tanpa berasumsi
print("jumlah SCHEDULED_DEPARTURE = '2400':", df_flights_p.filter(F.col("SCHEDULED_DEPARTURE") == "2400").count())
df_valid.agg(F.min("jam_jadwal").alias("jam_min"), F.max("jam_jadwal").alias("jam_max")).show()

Jika ada jam 24 atau nilai di luar 0-23, hasil pemeriksaan di atas menjadi dasar keputusan di Tahap 4 (preprocessing). Di Tahap 2 nilai itu tidak diubah, sehingga tampil sebagai kelompok sendiri pada tabel per jam.

### Operasi 2: Proporsi delay dan batal per maskapai (turunan 1)
`groupBy` per `AIRLINE`, `join` ke `airlines` untuk nama maskapai, lalu `orderBy`. Proporsi delay dihitung dari penerbangan valid, proporsi batal dari seluruh baris.

In [ ]:
# Agregasi delay per maskapai (penerbangan valid)
agg_delay = (df_valid.groupBy("AIRLINE")
             .agg(F.count(F.lit(1)).alias("total_valid"),
                  F.sum("is_delayed").alias("jumlah_delay")))

In [ ]:
# Agregasi batal per maskapai (semua baris)
agg_batal = (df_flights_p.groupBy("AIRLINE")
             .agg(F.count(F.lit(1)).alias("total_semua"),
                  F.sum("CANCELLED").alias("jumlah_batal")))

In [ ]:
# Menggabungkan kedua agregasi, join ke nama maskapai, hitung proporsi, urutkan dari delay tertinggi
df_nama_maskapai = df_airlines.withColumnRenamed("AIRLINE", "NAMA_MASKAPAI")

df_maskapai = (agg_delay
               .join(agg_batal, "AIRLINE")
               .join(df_nama_maskapai, F.col("AIRLINE") == F.col("IATA_CODE"))
               .withColumn("prop_delay", F.round(F.col("jumlah_delay") / F.col("total_valid"), 4))
               .withColumn("prop_batal", F.round(F.col("jumlah_batal") / F.col("total_semua"), 4))
               .select("AIRLINE", "NAMA_MASKAPAI", "total_valid", "jumlah_delay", "prop_delay",
                       "total_semua", "jumlah_batal", "prop_batal")
               .orderBy(F.desc("prop_delay")))
df_maskapai.show(truncate=False)

In [ ]:
# Selisih proporsi delay maskapai tertinggi dan terendah (dalam poin persentase)
df_selisih = df_maskapai.agg(F.max("prop_delay").alias("tertinggi"),
                             F.min("prop_delay").alias("terendah"))
baris_selisih = df_selisih.collect()[0]
print("delay tertinggi:", baris_selisih["tertinggi"])
print("delay terendah :", baris_selisih["terendah"])
print("selisih        :", round(baris_selisih["tertinggi"] - baris_selisih["terendah"], 4))

### Operasi 3: Pola waktu (turunan 2)
Proporsi delay per bulan, per hari dalam seminggu, dan per jam keberangkatan terjadwal. Pada `DAY_OF_WEEK`, nilai 1 adalah Senin (dicek dari sampel: 1 Januari 2015 hari Kamis bernilai 4).

In [ ]:
# Proporsi delay per bulan
df_bulan = (df_valid.groupBy("MONTH")
            .agg(F.count(F.lit(1)).alias("total_valid"),
                 F.sum("is_delayed").alias("jumlah_delay"))
            .withColumn("prop_delay", F.round(F.col("jumlah_delay") / F.col("total_valid"), 4))
            .orderBy("MONTH"))
df_bulan.show(12)

In [ ]:
# Proporsi delay per hari dalam seminggu
df_hari = (df_valid.groupBy("DAY_OF_WEEK")
           .agg(F.count(F.lit(1)).alias("total_valid"),
                F.sum("is_delayed").alias("jumlah_delay"))
           .withColumn("prop_delay", F.round(F.col("jumlah_delay") / F.col("total_valid"), 4))
           .orderBy("DAY_OF_WEEK"))
df_hari.show()

In [ ]:
# Proporsi delay per jam keberangkatan terjadwal
df_jam = (df_valid.groupBy("jam_jadwal")
          .agg(F.count(F.lit(1)).alias("total_valid"),
               F.sum("is_delayed").alias("jumlah_delay"))
          .withColumn("prop_delay", F.round(F.col("jumlah_delay") / F.col("total_valid"), 4))
          .orderBy("jam_jadwal"))
df_jam.show(30)

### Operasi 4: Join ke airports, proporsi delay per bandara asal, dan penyebab delay (turunan 3)
Tabel `airports` di-join dua kali (bandara asal dan tujuan) dengan **left join**, sehingga semua baris tetap ada. Kolom penanda `asal_terpetakan` dan `tujuan_terpetakan` menunjukkan apakah kode bandara ditemukan di `airports.csv`. Tidak ada data eksternal yang dipakai.

In [ ]:
# Memastikan IATA_CODE di airports unik, supaya join tidak menggandakan baris
print("baris airports       :", df_airports.count())
print("IATA_CODE yang unik  :", df_airports.select("IATA_CODE").distinct().count())

In [ ]:
# Menyiapkan tabel bandara asal dan bandara tujuan (nama kolom dibedakan)
df_asal = (df_airports
           .select(F.col("IATA_CODE").alias("ORIGIN_AIRPORT"),
                   F.col("AIRPORT").alias("NAMA_BANDARA_ASAL"),
                   F.col("STATE").alias("STATE_ASAL")))

df_tujuan = (df_airports
             .select(F.col("IATA_CODE").alias("DESTINATION_AIRPORT"),
                     F.col("AIRPORT").alias("NAMA_BANDARA_TUJUAN"),
                     F.col("STATE").alias("STATE_TUJUAN")))

In [ ]:
# Left join dua kali + kolom penanda terpetakan
df_valid_j = (df_valid
              .join(df_asal, "ORIGIN_AIRPORT", "left")
              .join(df_tujuan, "DESTINATION_AIRPORT", "left")
              .withColumn("asal_terpetakan", F.col("NAMA_BANDARA_ASAL").isNotNull())
              .withColumn("tujuan_terpetakan", F.col("NAMA_BANDARA_TUJUAN").isNotNull()))

# Jumlah baris harus tetap sama dengan df_valid (left join tidak boleh menambah/mengurangi baris)
print("baris df_valid  :", n_valid)
print("baris df_valid_j:", df_valid_j.count())

In [ ]:
# Jumlah baris valid yang kode bandaranya TIDAK terpetakan, per bulan
df_tak_terpetakan = (df_valid_j
                     .groupBy("MONTH")
                     .agg(F.count(F.lit(1)).alias("total_valid"),
                          F.sum(F.when(~F.col("asal_terpetakan"), 1).otherwise(0)).alias("asal_tak_terpetakan"),
                          F.sum(F.when(~F.col("tujuan_terpetakan"), 1).otherwise(0)).alias("tujuan_tak_terpetakan"))
                     .orderBy("MONTH"))
df_tak_terpetakan.show(12)

In [ ]:
# Total baris valid yang tidak terpetakan (untuk catatan keterbatasan)
df_valid_j.agg(
    F.sum(F.when(~F.col("asal_terpetakan"), 1).otherwise(0)).alias("asal_tak_terpetakan"),
    F.sum(F.when(~F.col("tujuan_terpetakan"), 1).otherwise(0)).alias("tujuan_tak_terpetakan"),
    F.sum(F.when(~F.col("asal_terpetakan") | ~F.col("tujuan_terpetakan"), 1).otherwise(0)).alias("asal_atau_tujuan_tak_terpetakan")
).show()

**Ambang jumlah penerbangan.** Proporsi dari sampel kecil tidak stabil. Galat baku proporsi kira-kira `sqrt(p(1-p)/n)`, yang paling besar saat p = 0,5: sekitar 5 poin persentase untuk n = 100, 2,2 poin untuk n = 500, dan 1,6 poin untuk n = 1.000. Karena itu:
- ranking **per bandara setahun** memakai minimal **1.000** penerbangan,
- ranking **per bandara per bulan** (window) memakai minimal **100** penerbangan,
- ranking **rute** setahun memakai minimal **500** penerbangan (rute lebih banyak dan lebih jarang terbang).

Angka ini keputusan analis, bukan standar baku, dan dicatat sebagai keterbatasan.

In [ ]:
# Proporsi delay per bandara asal (hanya bandara asal yang terpetakan, minimal 1000 penerbangan)
df_bandara_semua = (df_valid_j
                    .filter(F.col("asal_terpetakan"))
                    .groupBy("ORIGIN_AIRPORT", "NAMA_BANDARA_ASAL")
                    .agg(F.count(F.lit(1)).alias("total_valid"),
                         F.sum("is_delayed").alias("jumlah_delay"))
                    .withColumn("prop_delay", F.round(F.col("jumlah_delay") / F.col("total_valid"), 4)))

df_bandara = (df_bandara_semua
              .filter(F.col("total_valid") >= 1000)
              .orderBy(F.desc("prop_delay")))

print("bandara asal terpetakan        :", df_bandara_semua.count())
print("bandara yang lolos ambang 1000 :", df_bandara.count())
df_bandara.show(15, truncate=False)

In [ ]:
# Bandara asal dengan proporsi delay terendah (ambang yang sama)
df_bandara.orderBy("prop_delay").show(10, truncate=False)

In [ ]:
# Penyebab delay secara keseluruhan: total menit tiap penyebab pada penerbangan delay
df_sebab_total = (df_valid_j
                  .filter(F.col("is_delayed") == 1)
                  .agg(F.sum("AIR_SYSTEM_DELAY").alias("menit_air_system"),
                       F.sum("SECURITY_DELAY").alias("menit_security"),
                       F.sum("AIRLINE_DELAY").alias("menit_airline"),
                       F.sum("LATE_AIRCRAFT_DELAY").alias("menit_late_aircraft"),
                       F.sum("WEATHER_DELAY").alias("menit_weather"),
                       F.count(F.lit(1)).alias("baris_delay")))
df_sebab_total.show(truncate=False)

In [ ]:
# Persentase kontribusi tiap penyebab terhadap total menit
baris_sebab = df_sebab_total.collect()[0]
total_menit = (baris_sebab["menit_air_system"] + baris_sebab["menit_security"] +
               baris_sebab["menit_airline"] + baris_sebab["menit_late_aircraft"] +
               baris_sebab["menit_weather"])
for nama in ["menit_air_system", "menit_security", "menit_airline", "menit_late_aircraft", "menit_weather"]:
    print(nama, ":", round(100 * baris_sebab[nama] / total_menit, 2), "%")

In [ ]:
# Penyebab dominan di 10 bandara asal dengan proporsi delay tertinggi
df_sebab_bandara = (df_valid_j
                    .filter(F.col("asal_terpetakan") & (F.col("is_delayed") == 1))
                    .groupBy("ORIGIN_AIRPORT")
                    .agg(F.coalesce(F.sum("AIR_SYSTEM_DELAY"), F.lit(0)).alias("menit_air_system"),
                         F.coalesce(F.sum("SECURITY_DELAY"), F.lit(0)).alias("menit_security"),
                         F.coalesce(F.sum("AIRLINE_DELAY"), F.lit(0)).alias("menit_airline"),
                         F.coalesce(F.sum("LATE_AIRCRAFT_DELAY"), F.lit(0)).alias("menit_late_aircraft"),
                         F.coalesce(F.sum("WEATHER_DELAY"), F.lit(0)).alias("menit_weather")))

# Nilai menit terbesar di antara lima penyebab, lalu nama penyebabnya
menit_terbesar = F.greatest("menit_air_system", "menit_security", "menit_airline",
                            "menit_late_aircraft", "menit_weather")
df_sebab_bandara = df_sebab_bandara.withColumn(
    "penyebab_dominan",
    F.when(F.col("menit_air_system") == menit_terbesar, "AIR_SYSTEM")
     .when(F.col("menit_security") == menit_terbesar, "SECURITY")
     .when(F.col("menit_airline") == menit_terbesar, "AIRLINE")
     .when(F.col("menit_late_aircraft") == menit_terbesar, "LATE_AIRCRAFT")
     .otherwise("WEATHER"))

df_top_bandara = df_bandara.limit(10).join(df_sebab_bandara, "ORIGIN_AIRPORT").orderBy(F.desc("prop_delay"))
df_top_bandara.select("ORIGIN_AIRPORT", "NAMA_BANDARA_ASAL", "total_valid", "prop_delay",
                      "menit_air_system", "menit_security", "menit_airline",
                      "menit_late_aircraft", "menit_weather", "penyebab_dominan").show(truncate=False)

In [ ]:
# Rute (asal-tujuan) dengan proporsi delay tertinggi: hanya baris dengan asal DAN tujuan terpetakan, minimal 500 penerbangan
df_rute = (df_valid_j
           .filter(F.col("asal_terpetakan") & F.col("tujuan_terpetakan"))
           .groupBy("ORIGIN_AIRPORT", "DESTINATION_AIRPORT")
           .agg(F.count(F.lit(1)).alias("total_valid"),
                F.sum("is_delayed").alias("jumlah_delay"))
           .withColumn("prop_delay", F.round(F.col("jumlah_delay") / F.col("total_valid"), 4))
           .filter(F.col("total_valid") >= 500)
           .orderBy(F.desc("prop_delay")))
df_rute.show(10)

### Operasi 5: Window function, ranking bandara asal per bulan
Bandara asal diurutkan menurut proporsi delay **di dalam tiap bulan**. Ambang minimal 100 penerbangan per bandara per bulan (alasan di atas). Jika proporsi sama, bandara dengan penerbangan lebih banyak diutamakan, lalu urut kode agar hasil tetap.

In [ ]:
# Proporsi delay per bandara asal per bulan (minimal 100 penerbangan)
df_bandara_bulan = (df_valid_j
                    .filter(F.col("asal_terpetakan"))
                    .groupBy("MONTH", "ORIGIN_AIRPORT")
                    .agg(F.count(F.lit(1)).alias("total_valid"),
                         F.sum("is_delayed").alias("jumlah_delay"))
                    .withColumn("prop_delay", F.round(F.col("jumlah_delay") / F.col("total_valid"), 4))
                    .filter(F.col("total_valid") >= 100))

In [ ]:
# Window: peringkat dalam tiap bulan; tampilkan 5 teratas per bulan
jendela = Window.partitionBy("MONTH").orderBy(F.desc("prop_delay"), F.desc("total_valid"), F.asc("ORIGIN_AIRPORT"))

df_peringkat = (df_bandara_bulan
                .withColumn("peringkat", F.row_number().over(jendela))
                .filter(F.col("peringkat") <= 5)
                .orderBy("MONTH", "peringkat"))
df_peringkat.show(60)

## D. Pertanyaan yang dijawab dengan Spark SQL
Penerbangan valid (hasil join) didaftarkan sebagai temp view. Hasil SQL dicocokkan dengan versi DataFrame memakai `exceptAll` di kedua arah; jika hasilnya sama, kedua selisih berjumlah 0.

In [ ]:
# Mendaftarkan temp view untuk Spark SQL
df_valid_j.createOrReplaceTempView("flights_valid")
df_airlines.createOrReplaceTempView("airlines")

**Pertanyaan SQL 1 (turunan 1):** proporsi delay per maskapai.

In [ ]:
# Spark SQL: proporsi delay per maskapai
df_sql_maskapai = spark.sql("""
    SELECT f.AIRLINE,
           a.AIRLINE AS NAMA_MASKAPAI,
           COUNT(*) AS total_valid,
           SUM(f.is_delayed) AS jumlah_delay,
           ROUND(SUM(f.is_delayed) / COUNT(*), 4) AS prop_delay
    FROM flights_valid f
    JOIN airlines a ON f.AIRLINE = a.IATA_CODE
    GROUP BY f.AIRLINE, a.AIRLINE
    ORDER BY prop_delay DESC
""")
df_sql_maskapai.show(truncate=False)

In [ ]:
# Mencocokkan dengan versi DataFrame (df_maskapai)
df_versi_df = df_maskapai.select("AIRLINE", "NAMA_MASKAPAI", "total_valid", "jumlah_delay", "prop_delay")
print("baris di SQL tapi tidak di DataFrame:", df_sql_maskapai.exceptAll(df_versi_df).count())
print("baris di DataFrame tapi tidak di SQL:", df_versi_df.exceptAll(df_sql_maskapai).count())

**Pertanyaan SQL 2 (turunan 2):** proporsi delay per bulan.

In [ ]:
# Spark SQL: proporsi delay per bulan
df_sql_bulan = spark.sql("""
    SELECT MONTH,
           COUNT(*) AS total_valid,
           SUM(is_delayed) AS jumlah_delay,
           ROUND(SUM(is_delayed) / COUNT(*), 4) AS prop_delay
    FROM flights_valid
    GROUP BY MONTH
    ORDER BY MONTH
""")
df_sql_bulan.show(12)

In [ ]:
# Mencocokkan dengan versi DataFrame (df_bulan)
print("baris di SQL tapi tidak di DataFrame:", df_sql_bulan.exceptAll(df_bulan).count())
print("baris di DataFrame tapi tidak di SQL:", df_bulan.exceptAll(df_sql_bulan).count())

## E. Pengukuran waktu: CSV vs Parquet
Dua pipeline yang sama persis (filter penerbangan valid, `groupBy` maskapai, hitung total dan jumlah delay, `collect()`), hanya sumber datanya berbeda: CSV dengan schema eksplisit dan Parquet. Metode sama: `time.perf_counter()`, tanpa `.cache()`, 3 kali ulang, dan waktu pembacaan data ikut terhitung.

In [ ]:
# Pipeline 1: dari CSV, diulang 3 kali
waktu_csv = []
for i in range(3):
    mulai = time.perf_counter()
    df_uji = spark.read.csv("../data/flights.csv", header=True, schema=schema_flights)
    hasil_csv = (df_uji
                 .filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0) & F.col("ARRIVAL_DELAY").isNotNull())
                 .groupBy("AIRLINE")
                 .agg(F.count(F.lit(1)).alias("total_valid"),
                      F.sum(F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0)).alias("jumlah_delay"))
                 .collect())
    waktu_csv.append(time.perf_counter() - mulai)
print("waktu CSV (detik):", [round(w, 2) for w in waktu_csv])

In [ ]:
# Pipeline 2: dari Parquet, diulang 3 kali (transformasi dan action sama dengan pipeline 1)
waktu_parquet = []
for i in range(3):
    mulai = time.perf_counter()
    df_uji = spark.read.parquet("../data/processed/flights")
    hasil_parquet = (df_uji
                     .filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0) & F.col("ARRIVAL_DELAY").isNotNull())
                     .groupBy("AIRLINE")
                     .agg(F.count(F.lit(1)).alias("total_valid"),
                          F.sum(F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0)).alias("jumlah_delay"))
                     .collect())
    waktu_parquet.append(time.perf_counter() - mulai)
print("waktu Parquet (detik):", [round(w, 2) for w in waktu_parquet])

In [ ]:
# Memastikan hasil kedua pipeline sama, lalu merangkum waktu
print("hasil CSV sama dengan hasil Parquet:", sorted(hasil_csv) == sorted(hasil_parquet))
print("rata-rata CSV     :", round(sum(waktu_csv) / 3, 2), "detik")
print("rata-rata Parquet :", round(sum(waktu_parquet) / 3, 2), "detik")
print("waktu tercepat CSV     :", round(min(waktu_csv), 2), "detik")
print("waktu tercepat Parquet :", round(min(waktu_parquet), 2), "detik")

**Keterbatasan pengukuran**
- **Cache sistem operasi:** setelah file dibaca sekali, sebagian isinya tersimpan di memori OS, sehingga ulangan berikutnya biasanya lebih cepat tanpa `.cache()` Spark. Pipeline CSV dijalankan lebih dulu, jadi urutan ini bisa memengaruhi perbandingan.
- **Proses pertama lebih lambat:** run pertama menanggung pemanasan JVM, pembentukan rencana eksekusi, dan baca disk dingin. Karena itu 3 waktu ditampilkan, bukan satu angka.
- **I/O disk:** waktu bergantung pada jenis dan beban disk (SSD/HDD) saat pengujian. Parquet lebih kecil dan kolumnar, tetapi keunggulannya tidak otomatis besar pada 592 MB.
- **Hardware dan mode lokal:** Spark berjalan `local[*]` di satu laptop, jadi tidak ada overhead jaringan antar-node. Hasil tidak bisa diterapkan langsung ke klaster, dan program lain yang berjalan ikut memengaruhi waktu.
- **Hanya 3 pengulangan:** cukup untuk melihat variasi kasar, bukan untuk uji statistik. Waktu pembacaan data ikut terhitung di kedua pipeline.